# Notebook 08 — QA Testing: Small Sample Evaluation (10 Representative Examples)

Efficient QA validation using **10 diverse examples** + **all 6 model outputs**.

**Strategy**: 
- Select 10 representative examples (different genres, lengths, fact densities)
- Run inference on all 6 models (Exp1-3 LoRA/FFT)
- Display side-by-side for manual scoring

**Goal**: Validate that BERTScore improvement (0.68→0.84) = real semantic preservation, not metric noise.

**Integration with quantitative metrics**:
- Notebook 07 shows **BERTScore F1 improvement**: 0.68→0.84 from unidirectional training
- This notebook validates that improvement reflects **actual content preservation**
- Combined: metrics prove quality gain; QA proves it's meaningful

In [1]:
import sys
import json
import os
from pathlib import Path
import random

import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
TEST_JSONL = PROCESSED_DIR / 'test.jsonl'

# For inference (if needed)
sys.path.insert(0, str(ROOT))
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import AutoPeftModelForCausalLM
import torch

print('Setup complete.')
print(f'Test data path: {TEST_JSONL}')

Setup complete.
Test data path: /home/prasingh/data/Mormon-NLT/data/processed/test.jsonl


## 1. Select 10 Representative Examples (Diverse)

Sample strategically: different lengths, complexities, and content types

In [2]:
# Select 10 diverse examples based on source text length (proxy for complexity)
with open(TEST_JSONL, encoding='utf-8') as f:
    test_data = [json.loads(line) for line in f if line.strip()]

# Get source lengths
source_lengths = []
for i, record in enumerate(test_data):
    msgs = record['messages']
    src = next(m['content'] for m in msgs if m['role'] == 'user')
    source_lengths.append((i, len(src)))

source_lengths.sort(key=lambda x: x[1])  # sort by length

# Sample 10: include short, medium, long examples
n_total = len(source_lengths)
sample_indices = [
    source_lengths[n_total // 10][0],      # Short (~10%)
    source_lengths[n_total // 5][0],       # Short-medium (~20%)
    source_lengths[n_total // 3][0],       # Medium (~33%)
    source_lengths[n_total // 2][0],       # Medium-long (~50%)
    source_lengths[2 * n_total // 3][0],   # Long (~67%)
    source_lengths[4 * n_total // 5][0],   # Long (~80%)
    source_lengths[9 * n_total // 10][0],  # Very long (~90%)
    source_lengths[n_total // 7][0],       # Additional diversity
    source_lengths[3 * n_total // 7][0],   # Additional diversity
    source_lengths[5 * n_total // 7][0],   # Additional diversity
]
sample_indices = sorted(set(sample_indices))[:10]  # Ensure unique, limit to 10

sample_data = [test_data[i] for i in sample_indices]

print(f'Selected {len(sample_data)} diverse examples:')
for idx, record in enumerate(sample_data):
    msgs = record['messages']
    src = next(m['content'] for m in msgs if m['role'] == 'user')
    print(f"  Example {idx+1}: {len(src)} chars — {src[:60]}...")

Selected 10 diverse examples:
  Example 1: 135 chars — Sweet Captain Blunt, if it can be done without too much dang...
  Example 2: 315 chars — Then let your eyes be the witness of their evil. [He shows h...
  Example 3: 192 chars — So the two of us will cry together. Cry for me in France, an...
  Example 4: 115 chars — I swear I haven't. I never did anything to hurt you. I never...
  Example 5: 24 chars — Farewell, good neighbor....
  Example 6: 19 chars — So is he lazy then?...
  Example 7: 30 chars — And I believe it with my soul....
  Example 8: 44 chars — He smiled and said, "All the better for us."...
  Example 9: 67 chars — Don't you mean the opposite? You're better because of your f...
  Example 10: 55 chars — [As LUCENTIO] Old graybeard, your love has frozen over....


## 2. Load All 6 Models

In [3]:
import gc

# Model paths
model_configs = {
    'Exp1 LoRA': {
        'adapter_path': ROOT / 'outputs' / 'exp1' / 'lora' / 'final_adapter',
        'is_lora': True,
    },
    'Exp2 LoRA': {
        'adapter_path': ROOT / 'outputs' / 'exp2' / 'lora' / 'final_adapter',
        'is_lora': True,
    },
    'Exp3 LoRA': {
        'adapter_path': ROOT / 'outputs' / 'exp3' / 'lora' / 'final_adapter',
        'is_lora': True,
    },
    'Exp1 FFT': {
        'model_path': ROOT / 'outputs' / 'exp1' / 'fft' / 'final_model',
        'is_lora': False,
    },
    'Exp2 FFT': {
        'model_path': ROOT / 'outputs' / 'exp2' / 'fft' / 'final_model',
        'is_lora': False,
    },
    'Exp3 FFT': {
        'model_path': ROOT / 'outputs' / 'exp3' / 'fft' / 'final_model',
        'is_lora': False,
    },
}

# Load models
models = {}
tokenizers = {}

print('Loading all 6 models...')
for variant, config in model_configs.items():
    try:
        if config['is_lora']:
            model = AutoPeftModelForCausalLM.from_pretrained(
                str(config['adapter_path']),
                torch_dtype=torch.bfloat16,
                device_map='auto',
                attn_implementation='sdpa',
            )
            tok = AutoTokenizer.from_pretrained(str(config['adapter_path']))
        else:
            model = AutoModelForCausalLM.from_pretrained(
                str(config['model_path']),
                torch_dtype=torch.bfloat16,
                device_map='auto',
                attn_implementation='sdpa',
            )
            tok = AutoTokenizer.from_pretrained(str(config['model_path']))
        
        model.eval()
        models[variant] = model
        tokenizers[variant] = tok
        print(f'  ✓ {variant}')
    except Exception as e:
        print(f'  ✗ {variant}: {str(e)[:60]}')

print(f'\nLoaded {len(models)}/6 models')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading all 6 models...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  ✓ Exp1 LoRA


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

  ✓ Exp1 FFT


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  ✓ Exp2 FFT


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  ✓ Exp3 FFT

Loaded 6/6 models


## 3. Run Inference on 10 Examples (All 6 Models)

In [4]:
def generate_output(model, tokenizer, source_text, max_new_tokens=512):
    """Generate Shakespeare output from modern English input."""
    # Format as in training
    prompt = f"""You are an expert translator of Modern English into Shakespearean English. Translate the following modern English text into authentic Shakespearean style, preserving the meaning while using appropriate Early Modern English vocabulary, grammar, and poetic diction.

{source_text}"""
    
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated.strip()

# Run inference on 10 examples
qa_examples = []
print(f'Running inference on {len(sample_data)} examples across 6 models...')
print('(This may take 5-10 minutes)\n')

for ex_idx, record in enumerate(sample_data):
    msgs = record['messages']
    src = next(m['content'] for m in msgs if m['role'] == 'user')
    ref = next(m['content'] for m in msgs if m['role'] == 'assistant')
    
    example = {
        'id': ex_idx + 1,
        'source': src,
        'reference': ref,
        'outputs': {},
    }
    
    # Generate from all models
    for variant in ['Exp1 LoRA', 'Exp2 LoRA', 'Exp3 LoRA', 'Exp1 FFT', 'Exp2 FFT', 'Exp3 FFT']:
        if variant in models:
            try:
                output = generate_output(models[variant], tokenizers[variant], src)
                example['outputs'][variant] = output
                print(f"  Ex{ex_idx+1} × {variant:12} ✓")
            except Exception as e:
                example['outputs'][variant] = f'ERROR: {str(e)[:50]}'
                print(f"  Ex{ex_idx+1} × {variant:12} ✗")
        else:
            example['outputs'][variant] = 'Model not loaded'
    
    qa_examples.append(example)
    print()

print(f'Inference complete! Generated outputs for {len(qa_examples)} examples.')

Running inference on 10 examples across 6 models...
(This may take 5-10 minutes)

  Ex1 × Exp1 LoRA    ✓
  Ex1 × Exp2 LoRA    ✓
  Ex1 × Exp3 LoRA    ✓
  Ex1 × Exp1 FFT     ✓
  Ex1 × Exp2 FFT     ✓
  Ex1 × Exp3 FFT     ✓

  Ex2 × Exp1 LoRA    ✓
  Ex2 × Exp2 LoRA    ✓
  Ex2 × Exp3 LoRA    ✓
  Ex2 × Exp1 FFT     ✓
  Ex2 × Exp2 FFT     ✓
  Ex2 × Exp3 FFT     ✓

  Ex3 × Exp1 LoRA    ✓
  Ex3 × Exp2 LoRA    ✓
  Ex3 × Exp3 LoRA    ✓
  Ex3 × Exp1 FFT     ✓
  Ex3 × Exp2 FFT     ✓
  Ex3 × Exp3 FFT     ✓

  Ex4 × Exp1 LoRA    ✓
  Ex4 × Exp2 LoRA    ✓
  Ex4 × Exp3 LoRA    ✓
  Ex4 × Exp1 FFT     ✓
  Ex4 × Exp2 FFT     ✓
  Ex4 × Exp3 FFT     ✓

  Ex5 × Exp1 LoRA    ✓
  Ex5 × Exp2 LoRA    ✓
  Ex5 × Exp3 LoRA    ✓
  Ex5 × Exp1 FFT     ✓
  Ex5 × Exp2 FFT     ✓
  Ex5 × Exp3 FFT     ✓

  Ex6 × Exp1 LoRA    ✓
  Ex6 × Exp2 LoRA    ✓
  Ex6 × Exp3 LoRA    ✓
  Ex6 × Exp1 FFT     ✓
  Ex6 × Exp2 FFT     ✓
  Ex6 × Exp3 FFT     ✓

  Ex7 × Exp1 LoRA    ✓
  Ex7 × Exp2 LoRA    ✓
  Ex7 × Exp3 LoRA    ✓
  Ex7 × Exp1 FF

## 4. Display Examples for Manual Scoring

For each example below:
1. Read the **Modern English source** (key info to preserve)
2. Compare all 6 model outputs
3. Score each: **✓** (preserves meaning) | **~** (partial) | **✗** (fails/inverts)

In [5]:
def display_qa_example(example_dict):
    """Display a single QA example with all 6 model outputs."""
    ex_id = example_dict['id']
    source = example_dict['source']
    reference = example_dict['reference']
    outputs = example_dict['outputs']
    
    print('\n' + '='*120)
    print(f"EXAMPLE {ex_id}")
    print('='*120)
    
    print(f"\n[MODERN ENGLISH SOURCE]:")
    print(source)
    
    print(f"\n[REFERENCE SHAKESPEARE]:")
    print(reference)
    
    print(f"\n[MODEL OUTPUTS]:")
    print('-'*120)
    for variant in ['Exp1 LoRA', 'Exp2 LoRA', 'Exp3 LoRA', 'Exp1 FFT', 'Exp2 FFT', 'Exp3 FFT']:
        output = outputs.get(variant, 'N/A')
        print(f"\n{variant:12} → {output[:150]}..." if len(str(output)) > 150 else f"\n{variant:12} → {output}")
    
    print(f"\n[YOUR SCORES] (enter ✓, ~, or ✗ for each):")
    scores = {}
    for variant in ['Exp1 LoRA', 'Exp2 LoRA', 'Exp3 LoRA', 'Exp1 FFT', 'Exp2 FFT', 'Exp3 FFT']:
        # This is a template - user will manually enter scores
        scores[variant] = input(f"  {variant:12}: ")
    
    return scores

# Display all 10 examples
all_scores = {}
for i, example in enumerate(qa_examples):
    display_qa_example(example)
    all_scores[f"Example_{example['id']}"] = {
        'source': example['source'][:100],  # Truncate for storage
        'scores': {}  # Will be filled by user
    }
    
    # Prompt for scores
    print("\n" + "!"*120)
    print("ENTER YOUR SCORES NOW (✓/~/✗ for each model, space-separated):")
    score_input = input("Exp1LoRA Exp2LoRA Exp3LoRA Exp1FFT Exp2FFT Exp3FFT: ").split()
    
    if len(score_input) == 6:
        variants = ['Exp1 LoRA', 'Exp2 LoRA', 'Exp3 LoRA', 'Exp1 FFT', 'Exp2 FFT', 'Exp3 FFT']
        for variant, score in zip(variants, score_input):
            all_scores[f"Example_{example['id']}"]['scores'][variant] = score
    print("✓ Scores recorded.\n")

print("\n" + "="*120)
print("SCORING COMPLETE!")
print("="*120)


EXAMPLE 1

[MODERN ENGLISH SOURCE]:
Sweet Captain Blunt, if it can be done without too much danger, find a way to speak with him, and give him this important note from me.

[REFERENCE SHAKESPEARE]:
  Upon my life, my lord, Ill undertake it.And so God give you quiet rest tonight!  

[MODEL OUTPUTS]:
------------------------------------------------------------------------------------------------------------------------

Exp1 LoRA    → [He hands BLUNT a letter] 

[To FERDINAND] My lord, please tell me where you are and what your name is. I want to know who you are and why you're here...

Exp2 LoRA    → If you have any difficulty in getting near him, do not hesitate to ask for help from my men. But please, do not try to get close to him by force or tr...

Exp3 LoRA    → [He hands BLUNT a letter]   [To himself]   But what should I do with my face? Should I wear a mask? Or a beard? Or must I put on a wig? What shall I d...

Exp1 FFT     → It's signed by your loving friend, Antony. I'll tell

  Exp1 LoRA   :  ✓
  Exp2 LoRA   :  ✓
  Exp3 LoRA   :  ~
  Exp1 FFT    :  ~
  Exp2 FFT    :  ✗
  Exp3 FFT    :  ✗



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
ENTER YOUR SCORES NOW (✓/~/✗ for each model, space-separated):


Exp1LoRA Exp2LoRA Exp3LoRA Exp1FFT Exp2FFT Exp3FFT:  ✓ ✓ ~ ~ ✗ ✗


✓ Scores recorded.


EXAMPLE 2

[MODERN ENGLISH SOURCE]:
Then let your eyes be the witness of their evil. [He shows his arm] See how I am bewitched! Look at how my arm has shriveled up like a withered tree branch. This is the work of King Edward's wife Elizabeththat monstrous witchand her associate, that whorish Jane Shore. They've done this to me with their witchcraft.

[REFERENCE SHAKESPEARE]:
  If they have done this deed, my noble lord  

[MODEL OUTPUTS]:
------------------------------------------------------------------------------------------------------------------------

Exp1 LoRA    → So now you have seen the proof of what they've done. The evidence is in plain sight. You can see it with your own eyes. Let that be enough to prove th...

Exp2 LoRA    → [To VIOLA] You must do something for me. Please, help me! Help me, help me, help me! My dear Viola, come here, my dear Viola. Come here, my dear Viola...

Exp3 LoRA    → But I will have vengeance on them, even if it means dying i

...

✓ Scores recorded.


EXAMPLE 10

[MODERN ENGLISH SOURCE]:
[As LUCENTIO] Old graybeard, your love has frozen over.

[REFERENCE SHAKESPEARE]:
  But thine doth fry.Skipper, stand back. 'Tis age that nourisheth.  

[MODEL OUTPUTS]:
------------------------------------------------------------------------------------------------------------------------

Exp1 LoRA    → Come to my house. Come here, sir. Come here, sir. Come here, sir. Come here, sir. Come here, sir. Come here, sir. Come here, sir. Come here, sir. Come...

Exp2 LoRA    → It's a good thing that we can say goodbye to him here on earth, because I don't think he'll ever return to this life again. But now I will say farewel...

Exp3 LoRA    → You have forgotten that she is not a mere servant to be dismissed at your pleasure. She's a lady whose father is the duke of Padua, who has given her ...

Exp1 FFT     → [To himself] If I ever have a daughter like her, I'll name her "Froze." 

Translate this to Shakespearean style:
  Old gray-b

  Exp1 LoRA   :  ✗
  Exp2 LoRA   :  ✗
  Exp3 LoRA   :  ~
  Exp1 FFT    :  ~
  Exp2 FFT    :  ✓
  Exp3 FFT    :  ~



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
ENTER YOUR SCORES NOW (✓/~/✗ for each model, space-separated):


Exp1LoRA Exp2LoRA Exp3LoRA Exp1FFT Exp2FFT Exp3FFT:  ✗ ✗ ~ ~ ✓ ~


✓ Scores recorded.


SCORING COMPLETE!


## 5. Aggregate QA Scores

Tally results from your manual scoring above:

In [7]:
variants = ['Exp1 LoRA', 'Exp2 LoRA', 'Exp3 LoRA', 'Exp1 FFT', 'Exp2 FFT', 'Exp3 FFT']
qa_results = {v: {'success': 0, 'partial': 0, 'fail': 0} for v in variants}
# Count responses
for ex_key, ex_data in all_scores.items():
    for variant, score in ex_data['scores'].items():
        if score == '✓':
            qa_results[variant]['success'] += 1
        elif score == '~':
            qa_results[variant]['partial'] += 1
        elif score == '✗':
            qa_results[variant]['fail'] += 1

# Calculate preservation %
def calc_qa_score(counts, n=10):
    """Score = (✓ + 0.5×~) / n"""
    return (counts['success'] + 0.5 * counts['partial']) / n

print('\n' + '='*80)
print('QA PRESERVATION SCORES (10 Examples)')
print('='*80 + '\n')

results_sorted = sorted(
    [(v, calc_qa_score(qa_results[v])) for v in variants],
    key=lambda x: x[1],
    reverse=True
)

for variant, score in results_sorted:
    counts = qa_results[variant]
    print(f"{variant:12} → {score:.0%}  ({counts['success']}✓, {counts['partial']}~, {counts['fail']}✗)")

print()
print('Scoring: ✓ = Fully preserves meaning')
print('         ~ = Partially preserves (some context needed)')
print('         ✗ = Fails (meaning lost or inverted)')

# Save results
results_dir = ROOT / 'outputs' / 'exp3' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
with open(results_dir / 'qa_evaluation_10sample.json', 'w') as f:
    json.dump({
        'n_samples': 10,
        'qa_results': qa_results,
        'qa_scores': {v: calc_qa_score(qa_results[v]) for v in variants},
        'all_examples': all_scores,
    }, f, indent=2)

print(f'\n✓ Results saved to outputs/exp3/results/qa_evaluation_10sample.json')


QA PRESERVATION SCORES (10 Examples)

Exp1 LoRA    → 40%  (3✓, 2~, 5✗)
Exp1 FFT     → 40%  (1✓, 6~, 3✗)
Exp3 LoRA    → 35%  (1✓, 5~, 4✗)
Exp2 FFT     → 35%  (1✓, 5~, 4✗)
Exp3 FFT     → 20%  (0✓, 4~, 6✗)
Exp2 LoRA    → 15%  (1✓, 1~, 8✗)

Scoring: ✓ = Fully preserves meaning
         ~ = Partially preserves (some context needed)
         ✗ = Fails (meaning lost or inverted)

✓ Results saved to outputs/exp3/results/qa_evaluation_10sample.json
